# الدرس الثالث: التحكم بالتدفق والتنفيذ عبر invoke و stream و batch

## المقدمة والاهداف التعليمية
في هذا الدفتر، سنتعلم كيفية استخدام واجهة التنفيذ الموحدة (LangChain Runnable Interface). توفر هذه الواجهة ثلاث طرق اساسية لمعالجة الطلبات:
1. `invoke`: تنفيذ طلب مفرد والحصول على الاجابة كاملة دفعة واحدة.
2. `stream`: استقبال الرد مجزأ تدفقيا رمزا تلو الآخر (Token-by-Token) لتقليل زمن الانتظار الاولي (Time To First Token - TTFT).
3. `batch`: ارسال مجموعة من الطلبات المتزامنة مع التحكم في اقصى عدد للاتصالات المتوازية (Concurrency Control).

## الخطوة 1: اعداد بيئة العمل وتحميل النموذج
نقوم بتحميل النموذج مع تفعيل خاصية التدفق المباشر.

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

# تهيئة النموذج
model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0.3
)

## الخطوة 2: التنفيذ المفرد المتزامن عبر `invoke`
في هذا النمط، يقوم التطبيق بانتظار اكتمال توليد الرد بالكامل من الخادم قبل العودة بالنتيجة. يعتبر هذا النمط مثاليا للمهام الخلفية (Background Jobs) ومعالجة البيانات غير التفاعلية.

In [ ]:
prompt_text = "Summarize the primary benefits of unit testing in software development in 3 bullet points."
response = model.invoke(prompt_text)

print("Synchronous Response:")
print(response.content)

## الخطوة 3: التوليد التدفقي عبر `stream`
يسمح التوليد التدفقي بعرض الرد للمستخدم فور توليد كل مقطع (Chunk)، مما يحسن تجربة المستخدم التفاعلية (User Experience) بشكل جذري.

In [ ]:
prompt_stream = "Explain what an API is in 2 concise sentences."

print("Streaming Output:")
for chunk in model.stream(prompt_stream):
    # طباعة كل مقطع فور وصوله دون اضافة سطر جديد مع تفريغ المخزن المؤقت
    print(chunk.content, end="", flush=True)
print("\n\n[Streaming Completed]")

## الخطوة 4: المعالجة المجمعة المتوازية عبر `batch`
تتيح دالة `batch` ارسال قائمة كاملة من المدخلات. الميزة الاهم في الاصدارات الحديثة هي القدرة على تحديد معامل `max_concurrency` داخل كائن الاعدادات `config` لتفادي تجاوز حدود الاستهلاك (Rate Limits) للمزود.

In [ ]:
prompts_list = [
    "Define Machine Learning in one sentence.",
    "Define Deep Learning in one sentence.",
    "Define Reinforcement Learning in one sentence."
]

# تنفيذ الطلبات الثلاثة بالتوازي بحد اقصى اتصالين في الوقت نفسه
batch_responses = model.batch(
    prompts_list,
    config={"max_concurrency": 2}
)

print("Batch Processing Results:")
for idx, res in enumerate(batch_responses, start=1):
    print(f"Task {idx}:")
    print(res.content.strip())
    print("-" * 40)